# Course grades
Run top to bottom. Nothing is emailed until the last cell, and only with `really_send=True`.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import pandas as pd
import gradetools as gt

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 80)

## 1. Pick the course
Each course is a dict in `courses/`. Edit it, then re-run from here.

In [ ]:
from courses.phys310_f26 import COURSE

cfg = gt.load_config(COURSE)
# cfg['gradebook'] = 'some_other_export.csv'   # or point at a newer export

## 2. Load and check
Read the report: UNCLASSIFIED, AMBIGUOUS and partly-graded assignments are the things to fix in the config.

In [ ]:
gb = gt.load_canvas(cfg['gradebook'], skip_students=cfg['skip_students'])
cls = gt.classify(gb, cfg)
issues = gt.diagnose(gb, cls, cfg)

assert not [m for lvl, m in issues if lvl == 'ERROR']

In [ ]:
# Every assignment column and what happened to it
gt.assignment_table(cls)

## 3. Compute grades

In [ ]:
res = gt.compute(gb, cls, cfg)
res.summary()

### Cross-check against Canvas
For categories with `canvas_group` set, recomputes each category the way Canvas does (total points, blanks skipped, no drops) and compares with Canvas's own column. `(course)` compares the weighted total with Canvas's Current Score, which tests your weights. A MISMATCH means this config and Canvas disagree about which assignments are in a group, or about the weights.

In [ ]:
gt.canvas_check(gb, cls, cfg)

## 4. Class plots (for you)

In [ ]:
fig = gt.class_overview(res)

In [ ]:
# Who would be affected most by drops, lowest grades first, etc.
res.summary().sort_values('grade')#.head(10)

## 5. Build the emails
Writes one figure + text body per student and a `manifest.csv` into the outbox folder. Re-running keeps the record of what was already sent.

In [ ]:
outbox = gt.build_outbox(res, gb)

In [ ]:
gt.preview(outbox, 'Kaupelis')   # a name, email, or Canvas ID

## 6. Send
1. Dry run (lists what would go out).
2. Send one or two to yourself with `test_to=` and check how they look in your inbox.
3. Send for real. Each successful send is recorded in the manifest, so re-running skips those students.

Password: set the environment variable `GRADETOOLS_SMTP_PASSWORD`, or you'll be prompted. For Gmail this must be an App Password.

In [ ]:
gt.send_emails(outbox, cfg)   # dry run

In [ ]:
#gt.send_emails(outbox, cfg, test_to='mbellis@siena.edu', only=['Oechslin'], really_send=True)
#gt.send_emails(outbox, cfg, only=['Oechslin'], really_send=True, resend=True)

In [ ]:
#gt.send_emails(outbox, cfg, really_send=True)